In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report, roc_auc_score

In [4]:
import pandas as pd

df = pd.read_csv('dataset/dataset_updated.csv')

print(df.head())
print(df.info())

                                               URL  url_length  \
0  https://keraekken-loagginnusa.godaddysites.com/          47   
1         https://metamsk01lgiix.godaddysites.com/          40   
2                          http://myglobaltech.in/          23   
3                   http://djtool-for-spotify.com/          30   
4  https://scearmcoommunnlty.com/invent/freind/get          47   

   has_ip_address  dot_count  https_flag  url_entropy  token_count  \
0               0          2           1     4.250669            6   
1               0          2           1     4.196439            6   
2               0          1           0     3.936180            5   
3               0          1           0     3.894740            5   
4               0          1           1     4.143127            7   

   subdomain_count  query_param_count  tld_length  path_length  \
0                1                  1           3            1   
1                1                  1           3 

In [7]:
df = df.dropna()
df = df.drop_duplicates()

df['URL'] = df['URL'].astype(str)
df['ClassLabel'] = df['ClassLabel'].astype(int)

In [8]:
import re
import numpy as np
from urllib.parse import urlparse

def extract_features(url):
    parsed = urlparse(url)
    url_length = len(url)

    has_ip_address = 1 if re.search(r'\d+\.\d+\.\d+\.\d+', url) else 0
    dot_count = url.count('.')
    https_flag = 1 if parsed.scheme == "https" else 0

    prob = [float(url.count(c)) / len(url) for c in set(url)]
    url_entropy = -sum([p * np.log2(p) for p in prob])

    tokens = re.split(r'[./\-?=&]', url)
    token_count = len([t for t in tokens if t])

    parts = parsed.netloc.split('.')
    subdomain_count = len(parts) - 2 if len(parts) > 2 else 0

    query_param_count = len(parsed.query.split('&')) if parsed.query else 0

    tld = parts[-1] if parts else ''
    tld_length = len(tld)

    path_length = len(parsed.path)

    domain = parsed.netloc
    has_hyphen_in_domain = 1 if '-' in domain else 0

    number_of_digits = sum(c.isdigit() for c in url)

    popular_tlds = ['com', 'org', 'net', 'edu', 'gov']
    tld_popularity = 1 if tld.lower() in popular_tlds else 0

    suspicious_file_extension = 1 if parsed.path.lower().endswith(('.exe','.zip','.rar')) else 0

    domain_name_length = len(domain)
    percentage_numeric_chars = number_of_digits / url_length if url_length > 0 else 0

    phishing_keywords = ['login','verify','secure','account','bank','confirm','password']
    has_phishing_keyword = 1 if any(k in url.lower() for k in phishing_keywords) else 0

    trusted_domains = ['youtube.com','google.com','facebook.com']
    is_trusted_domain = 1 if any(td in domain.lower() for td in trusted_domains) else 0

    return [
        url_length, has_ip_address, dot_count, https_flag, url_entropy,
        token_count, subdomain_count, query_param_count,
        tld_length, path_length, has_hyphen_in_domain,
        number_of_digits, tld_popularity, suspicious_file_extension,
        domain_name_length, percentage_numeric_chars,
        has_phishing_keyword, is_trusted_domain
    ]

In [9]:
feature_names = [
    "url_length","has_ip_address","dot_count","https_flag","url_entropy",
    "token_count","subdomain_count","query_param_count",
    "tld_length","path_length","has_hyphen_in_domain",
    "number_of_digits","tld_popularity","suspicious_file_extension",
    "domain_name_length","percentage_numeric_chars",
    "has_phishing_keyword","is_trusted_domain"
]

X = pd.DataFrame(df['URL'].apply(extract_features).tolist(), columns=feature_names)
y = df['ClassLabel']

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

dt = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
lr = LogisticRegression(max_iter=1000)

dt.fit(X_train, y_train)
rf.fit(X_train, y_train)
lr.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [13]:
from sklearn.metrics import accuracy_score, confusion_matrix

for name, model in [("DT",dt),("RF",rf)]:
    pred = model.predict(X_test)
    print(name, accuracy_score(y_test, pred))
    print(confusion_matrix(y_test, pred))

pred_lr = lr.predict(X_test_scaled)
print("LR", accuracy_score(y_test, pred_lr))
print(confusion_matrix(y_test, pred_lr))

DT 0.9977722772277228
[[12679    27]
 [   18  7476]]
RF 0.9989108910891089
[[12703     3]
 [   19  7475]]
LR 0.9956930693069307
[[12642    64]
 [   23  7471]]


In [16]:
import joblib

joblib.dump(rf, 'model/rf_modelnewfitur.pkl')
joblib.dump(scaler, 'model/scaler.pkl')

['model/scaler.pkl']

In [18]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred)
print(cm)

[[12703     3]
 [   19  7475]]


In [19]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     12706
           1       1.00      1.00      1.00      7494

    accuracy                           1.00     20200
   macro avg       1.00      1.00      1.00     20200
weighted avg       1.00      1.00      1.00     20200



In [21]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

# split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# train
model.fit(X_train, y_train)

# predict
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# accuracy
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)